# Notebook for running the multiple predictive models for determening if a mushroom is edible or poisonous. 

In [ ]:
# General Imports 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random

# scikit-learn imports
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import LabelEncoder

from xgboost import XGBClassifier # An external classifier

# Fetching dataset
from ucimlrepo import fetch_ucirepo  
mushroom = fetch_ucirepo(id=73) 

The dataset used is composed of the visual descriptive caracteristics of hypothetical samples corresponding to 23 species of gilled mushrooms in the Agaricus and Lepiota Family.

# ETL of data
below we extract the data we need it, randomly split it into train and test, build the encoding algorythm necessary for preprocessing the categorical data, and prepare to load it into the ML pipeline.

In [ ]:
# Raw data (as pandas dataframes) 
X = mushroom.data.features 
y = mushroom.data.targets 

# Setting up the encoder and preprocessor for the data
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
preprocessor = ColumnTransformer(
    transformers=[
        ('Categories', encoder, X.columns)
    ],
)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# Initialize the LabelEncoder
le = LabelEncoder()

# Flatten (ravel) the target arrays and encode them to 0s and 1s
y_train = le.fit_transform(np.ravel(y_train))
y_test = le.transform(np.ravel(y_test))

# Verify the mapping (Optional, but good to know which is which!)
print("Target mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

# variable information 
print(mushroom.variables) 
print(X_train.shape)
print(X_test.shape)
X_train.head()

Target mapping: {'e': np.int64(0), 'p': np.int64(1)}
                        name     role         type demographic  \
0                  poisonous   Target  Categorical        None   
1                  cap-shape  Feature  Categorical        None   
2                cap-surface  Feature  Categorical        None   
3                  cap-color  Feature       Binary        None   
4                    bruises  Feature  Categorical        None   
5                       odor  Feature  Categorical        None   
6            gill-attachment  Feature  Categorical        None   
7               gill-spacing  Feature  Categorical        None   
8                  gill-size  Feature  Categorical        None   
9                 gill-color  Feature  Categorical        None   
10               stalk-shape  Feature  Categorical        None   
11                stalk-root  Feature  Categorical        None   
12  stalk-surface-above-ring  Feature  Categorical        None   
13  stalk-surface-below

,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,stalk-shape,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
7873,k,s,e,f,s,f,c,n,b,t,...,k,p,w,p,w,o,e,w,v,d
6515,x,s,n,f,f,f,c,n,b,t,...,s,w,w,p,w,o,e,w,v,p
6141,f,y,e,f,y,f,c,n,b,t,...,s,p,w,p,w,o,e,w,v,l
2764,f,f,n,t,n,f,c,b,u,t,...,s,g,p,p,w,o,p,n,v,d
438,b,y,y,t,l,f,c,b,k,e,...,s,w,w,p,w,o,p,n,n,m


# Pipeline for comparing multiple predictive models

In [12]:
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('scaler', StandardScaler()), # Crucial for KNN and Logistic Regression
    ('classifier', LogisticRegression()) # A placeholder for the classifier so GridSearch can swap models
])

# --- HYPERPARAMETER GRID ---
# Note the 'classifier' key swaps the model type entirely
param_grid = [
    {
        'classifier': [LogisticRegression(max_iter=1000)],
        'classifier__C': [0.1, 1.0, 10.0],
        'classifier__penalty': ['l2']
    },
    {
        'classifier': [RandomForestClassifier(random_state=42)],
        'classifier__n_estimators': [100, 200],
        'classifier__max_depth': [None, 10]
    },
    {
        'classifier': [XGBClassifier(eval_metric='logloss', random_state=42)],
        'classifier__learning_rate': [0.05, 0.1],
        'classifier__max_depth': [3, 6]
    },
    {
        'classifier': [KNeighborsClassifier()],
        'classifier__n_neighbors': [5, 11, 21],
        'classifier__weights': ['uniform', 'distance']
    }
]

In [13]:
print("Starting Grid Search across 4 models...")
grid_search = GridSearchCV(
    full_pipeline, 
    param_grid, 
    cv=5, 
    scoring='accuracy', 
    n_jobs=-1, 
    verbose=1
)

grid_search.fit(X_train, y_train)

Starting Grid Search across 4 models...
Fitting 5 folds for each of 17 candidates, totalling 85 fits


c:\Users\User\anaconda3\envs\PM_project\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...egression())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'classifier': [LogisticRegre...max_iter=1000)], 'classifier__C': [0.1, 1.0, ...], 'classifier__penalty': ['l2']}, {'classifier': [RandomForestC...ndom_state=42)], 'classifier__max_depth': [None, 10], 'classifier__n_estimators': [100, 200]}, ...]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,

# Results

In [14]:
print("-" * 30)
print(f"Best Model Type: {type(grid_search.best_params_['classifier']).__name__}")
print(f"Best Accuracy: {grid_search.best_score_:.4f}")
print(f"Best Parameters: {grid_search.best_params_}")
print("-" * 30)

# Evaluate on the unseen test set
test_score = grid_search.score(X_test, y_test)
print(f"Final Test Set Accuracy: {test_score:.4f}")

------------------------------
Best Model Type: LogisticRegression
Best Accuracy: 1.0000
Best Parameters: {'classifier': LogisticRegression(max_iter=1000), 'classifier__C': 1.0, 'classifier__penalty': 'l2'}
------------------------------
Final Test Set Accuracy: 1.0000


Given that we have succesfully done a perfect fit, a further process of our data will be done to trial how many relevant characteristics we can drop while still predicting succesfully if a mushroom is edible.

To best challange the chosen models, the feature dropping will follow an importance criteria, where we will drop the most relevant feature across all models and retrain to see how much worse can these models work.

In [15]:

# Start with all original features
current_features = list(X.columns)
dropped_features = []
accuracy = 1.0

print("Starting Feature Drop...")

# Continue looping as long as we have a perfect fit and at least 1 feature remains
while accuracy >= 0.95 and len(current_features) > 1:
    print(f"\n--- Running with {len(current_features)} features ---")
    
    # 1. Subset the training and testing data
    X_train_sub = X_train[current_features]
    X_test_sub = X_test[current_features]
    
    # 2. Dynamically update the preprocessor so it only expects the remaining columns
    current_preprocessor = ColumnTransformer(
        transformers=[
            ('Categories', OneHotEncoder(handle_unknown='ignore', sparse_output=False), current_features)
        ]
    )
    
    # 3. Rebuild the pipeline with the updated preprocessor
    current_pipeline = Pipeline(steps=[
        ('preprocessor', current_preprocessor),
        ('scaler', StandardScaler()), 
        ('classifier', LogisticRegression()) # Placeholder for grid search
    ])
    
    # 4. Run the Grid Search 
    grid_search = GridSearchCV(
        current_pipeline, 
        param_grid, 
        cv=5, 
        scoring='accuracy', 
        n_jobs=-1, 
        verbose=0 # Set to 0 to keep the loop output clean
    )
    grid_search.fit(X_train_sub, y_train)
    
    # 5. Evaluate the best model found
    best_model = grid_search.best_estimator_
    accuracy = grid_search.score(X_test_sub, y_test)
    
    print(f"Best Model: {type(best_model.named_steps['classifier']).__name__}")
    print(f"Test Accuracy: {accuracy:.4f}")
    
    # Break the loop if the model can no longer perfectly predict
    if accuracy < 0.95:
        print("\nAccuracy dropped below 95%. Stopping!")
        break
        
    # 6. Calculate Permutation Importance on the test set
    # This evaluates which original column causes the biggest accuracy drop when shuffled
    result = permutation_importance(
        best_model, X_test_sub, y_test, n_repeats=5, random_state=42, n_jobs=-1
    )
    
    # 7. Identify the feature with the highest importance (biggest influence)
    most_important_idx = result.importances_mean.argmax()
    most_important_feature = current_features[most_important_idx]
    
    print(f"Most influential feature found: '{most_important_feature}' (Importance score: {result.importances_mean[most_important_idx]:.4f})")
    print(f"Dropping '{most_important_feature}'...")
    
    # 8. Remove the most important feature for the next iteration
    current_features.remove(most_important_feature)
    dropped_features.append(most_important_feature)

print("\n" + "="*40)
print("FINAL RESULTS")
print("="*40)
print(f"Total features dropped sequentially: {len(dropped_features)}")
print(f"Order of dropped features: {dropped_features}")
print(f"Remaining features: {current_features}")
print(f"Final accuracy achieved: {accuracy:.4f}")

Starting Feature Drop...

--- Running with 22 features ---


c:\Users\User\anaconda3\envs\PM_project\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Best Model: LogisticRegression
Test Accuracy: 1.0000
Most influential feature found: 'odor' (Importance score: 0.0790)
Dropping 'odor'...

--- Running with 21 features ---


c:\Users\User\anaconda3\envs\PM_project\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Best Model: LogisticRegression
Test Accuracy: 1.0000
Most influential feature found: 'spore-print-color' (Importance score: 0.1255)
Dropping 'spore-print-color'...

--- Running with 20 features ---


c:\Users\User\anaconda3\envs\PM_project\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Best Model: LogisticRegression
Test Accuracy: 1.0000
Most influential feature found: 'stalk-root' (Importance score: 0.1728)
Dropping 'stalk-root'...

--- Running with 19 features ---
Best Model: RandomForestClassifier
Test Accuracy: 1.0000
Most influential feature found: 'gill-size' (Importance score: 0.0869)
Dropping 'gill-size'...

--- Running with 18 features ---
Best Model: RandomForestClassifier
Test Accuracy: 0.9969
Most influential feature found: 'population' (Importance score: 0.0351)
Dropping 'population'...

--- Running with 17 features ---
Best Model: RandomForestClassifier
Test Accuracy: 0.9957
Most influential feature found: 'habitat' (Importance score: 0.0274)
Dropping 'habitat'...

--- Running with 16 features ---
Best Model: XGBClassifier
Test Accuracy: 0.9877
Most influential feature found: 'stalk-surface-above-ring' (Importance score: 0.2260)
Dropping 'stalk-surface-above-ring'...

--- Running with 15 features ---
Best Model: XGBClassifier
Test Accuracy: 0.9889
Most 

In [16]:
print(pd.DataFrame(grid_search.cv_results_)[['param_classifier', 'mean_test_score']])

                                     param_classifier  mean_test_score
0                   LogisticRegression(max_iter=1000)         0.898445
1                   LogisticRegression(max_iter=1000)         0.899060
2                   LogisticRegression(max_iter=1000)         0.899060
3             RandomForestClassifier(random_state=42)         0.916142
4             RandomForestClassifier(random_state=42)         0.916295
5             RandomForestClassifier(random_state=42)         0.908139
6             RandomForestClassifier(random_state=42)         0.903986
7   XGBClassifier(base_score=None, booster=None, c...         0.874442
8   XGBClassifier(base_score=None, booster=None, c...         0.902293
9   XGBClassifier(base_score=None, booster=None, c...         0.887828
10  XGBClassifier(base_score=None, booster=None, c...         0.919063
11                             KNeighborsClassifier()         0.913218
12                             KNeighborsClassifier()         0.915218
13    